In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Topic Subject

### CQ4.01u - Which SOSA sensors publish to which MQTT Topics?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?sensor ?topic WHERE {
  ?sensor a sosa:Sensor ;
          mqtt:observesTopic ?topic .
  ?topic a mqtt:Topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ4.02u - Which SOSA actuators listen to which MQTT Topics?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?actuator ?topic WHERE {
  ?actuator a sosa:Actuator ;
            mqtt:listensToTopic ?topic .
  ?topic a mqtt:Topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ4.03u - Which SOSA FeaturesOfInterest, Properties etc. appear as MQTT TopicSubjects and on which MQTT Topics?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX sosa: <http://www.w3.org/ns/sosa/> 

SELECT ?topic ?subject ?type WHERE {
  ?topic a mqtt:Topic ;
         mqtt:hasSubject ?subject .
  ?subject a ?type .
  FILTER( STRSTARTS(STR(?type), STR(sosa:)) )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ4.04u - Which MQTT Topics are stored?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?topic WHERE {
  ?topic a mqtt:Topic ;
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short